In [7]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# 🔍 ฟังก์ชันค้นหาไฟล์อัตโนมัติ
def locate_file(filename):
    # ลองค้นหาในโฟลเดอร์ปัจจุบัน และโฟลเดอร์ยอดนิยมของ Platform
    search_dirs = [".", "./data", "../input", "/kaggle/input", "/workspace"]
    for d in search_dirs:
        if os.path.exists(d):
            matches = glob.glob(f"{d}/**/{filename}", recursive=True)
            if matches:
                return matches[0]
    raise FileNotFoundError(f"🚨 ไม่พบไฟล์ '{filename}' โปรดดาวน์โหลดจากหน้า Competition แล้ววางในโฟลเดอร์เดียวกับโค้ด")

print("🔎 กำลังค้นหาไฟล์ข้อมูล...")
train_path = locate_file("train.csv")
test_path = locate_file("test.csv")
sub_path = locate_file("sample_submission.csv")

print(f"✅ พบ train.csv: {train_path}")
print(f"✅ พบ test.csv: {test_path}")
print(f"✅ พบ sample_submission.csv: {sub_path}")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_sub = pd.read_csv(sub_path)

# 🛠 ตั้งค่าคงที่
TARGET = 'History of HeartDisease or Attack'
ID_COL = 'ID'
SEED = 42

print("📥 กำลังโหลดข้อมูล...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 🔍 รวมข้อมูลเพื่อทำ Preprocessing พร้อมกัน (ป้องกัน Train/Test mismatch)
train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan
df = pd.concat([train, test], ignore_index=True)

# 🧹 1. Preprocessing พื้นฐาน
# แปลง Yes/No -> 1/0 (ครอบคลุมตัวพิมพ์ใหญ่/เล็ก และ NaN)
yes_no_cols = [c for c in df.columns if c not in [ID_COL, TARGET, 'is_train', 'Age', 'Body Mass Index', 'General Health', 'Education Level', 'Income Level', 'Sex']]
for col in yes_no_cols:
    if col in df.columns:
        df[col] = df[col].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0, 1.0: 1, 0.0: 0})

# แปลงเพศ
df['Sex'] = df['Sex'].map({'Male': 1, 'Female': 0, 'male': 1, 'female': 0, 'M': 1, 'F': 0}).fillna(0)

# แปลง Ordinal/Categorical (ใช้ category codes เพื่อประหยัด memory และคงลำดับ)
for col in ['General Health', 'Education Level', 'Income Level']:
    if col in df.columns:
        df[col] = pd.Categorical(df[col].astype(str)).codes  # NaN จะถูกแปลงเป็น -1 ซึ่ง HistGBDT รับมือได้

# 🛠 2. Feature Engineering เฉพาะทางสุขภาพ
if 'Age' in df.columns and 'Body Mass Index' in df.columns:
    df['BMI_x_Age'] = df['Body Mass Index'] * df['Age']
    df['Age_Group'] = pd.cut(df['Age'], bins=[0, 30, 45, 60, 100], labels=[0, 1, 2, 3]).astype(float).fillna(0)

# คำนวณคะแนนโรคประจำตัวรวม (Comorbidity Score)
risk_conditions = ['High Blood Pressure', 'Diagnosed Stroke', 'Diagnosed Diabetes', 'Told High Cholesterol']
available_risk = [c for c in risk_conditions if c in df.columns]
df['Comorbidity_Score'] = df[available_risk].sum(axis=1)

# 📦 แยกข้อมูลกลับ
train_df = df[df['is_train'] == 1].copy()
test_df = df[df['is_train'] == 0].copy()

feature_cols = [c for c in df.columns if c not in [ID_COL, TARGET, 'is_train']]
X_train = train_df[feature_cols]
y_train = train_df[TARGET].map({'Yes': 1, 'No': 0, 1: 1, 0: 0}).fillna(0).astype(int)
X_test = test_df[feature_cols]

print(f"✅ Preprocessing เสร็จ | Features: {len(feature_cols)} | Train: {len(X_train)} | Test: {len(X_test)}")
print(f"🎯 สัดส่วน Class: {y_train.value_counts(normalize=True).to_dict()}\n")

# 🎯 3. ฟังก์ชันหา Threshold ที่ดีที่สุดสำหรับ F2-Score
def find_optimal_threshold(y_true, y_proba, beta=2, low=0.1, high=0.6, step=0.005):
    best_f2, best_th = 0, 0.5
    for th in np.arange(low, high, step):
        preds = (y_proba >= th).astype(int)
        f2 = fbeta_score(y_true, preds, beta=beta, zero_division=0)
        if f2 > best_f2:
            best_f2, best_th = f2, th
    return best_th, best_f2

# 🔄 4. Cross-Validation + Threshold Tuning
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_proba = np.zeros(len(X_train))
fold_f2 = []

print("🔍 กำลังทำ Stratified 5-Fold CV...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    # HistGradientBoosting รองรับ NaN โดยธรรมชาติ + เร็ว + ไม่ต้องการ Scaling
    model = HistGradientBoostingClassifier(
        max_iter=400,
        learning_rate=0.03,
        max_depth=5,
        min_samples_leaf=30,
        l2_regularization=0.05,
        class_weight='balanced',  # ช่วยดึง Recall ขึ้น สอดคล้องกับ F2
        random_state=SEED
    )
    model.fit(X_tr, y_tr)
    
    val_proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_idx] = val_proba

    th, f2 = find_optimal_threshold(y_val, val_proba)
    fold_f2.append(f2)
    print(f"📌 Fold {fold+1} | Threshold: {th:.3f} | F2: {f2:.4f}")

overall_th, overall_f2 = find_optimal_threshold(y_train, oof_proba)
print(f"\n📈 Mean CV F2: {np.mean(fold_f2):.4f} | Overall Optimal Th: {overall_th:.3f} | F2: {overall_f2:.4f}")

# 🚀 5. Train Final Model บนข้อมูล Train ทั้งหมด
print("\n🤖 กำลัง Train Final Model...")
final_model = HistGradientBoostingClassifier(
    max_iter=500,
    learning_rate=0.025,
    max_depth=6,
    min_samples_leaf=25,
    l2_regularization=0.03,
    class_weight='balanced',
    random_state=SEED
)
final_model.fit(X_train, y_train)

# 🔮 ทำนาย Test Set
test_proba = final_model.predict_proba(X_test)[:, 1]
test_preds = (test_proba >= overall_th).astype(int)

# 📄 6. สร้างไฟล์ Submission
# 🔹 แปลง Array 0/1 เป็น 'No'/'Yes' แบบ Vectorized (เร็วและไม่มี Error)
test_labels = np.where(test_preds == 1, 'Yes', 'No')

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_labels
})
submission.to_csv('submission.csv', index=False)
print(f"✅ บันทึก submission.csv เรียบร้อย! (พยากรณ์ Yes: {(test_preds==1).sum()} คน)")

🔎 กำลังค้นหาไฟล์ข้อมูล...
✅ พบ train.csv: ../input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/train.csv
✅ พบ test.csv: ../input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/test.csv
✅ พบ sample_submission.csv: ../input/competitions/super-ai-engineer-ss-6-heart-disease-prediction/sample_submission.csv
✅ Preprocessing เสร็จ | Features: 21 | Train: 223084 | Test: 74361
🎯 สัดส่วน Class: {0: 0.91900808664001, 1: 0.08099191335998995}

🔍 กำลังทำ Stratified 5-Fold CV...
📌 Fold 1 | Threshold: 0.595 | F2: 0.5436
📌 Fold 2 | Threshold: 0.565 | F2: 0.5310
📌 Fold 3 | Threshold: 0.545 | F2: 0.5373
📌 Fold 4 | Threshold: 0.590 | F2: 0.5444
📌 Fold 5 | Threshold: 0.585 | F2: 0.5397

📈 Mean CV F2: 0.5392 | Overall Optimal Th: 0.550 | F2: 0.5382

🤖 กำลัง Train Final Model...
✅ บันทึก submission.csv เรียบร้อย! (พยากรณ์ Yes: 23270 คน)
